> **Meridian AgentOps workshop · notebook 04 of 6 (Modules 7–8).** The reusable code lives in the
> repo's `app/` package (agent, tools, MCP service desk, evaluators, config) — these notebooks
> import it, so a fresh session only needs the bootstrap cells below instead of re-running
> earlier modules. **Prerequisite:** notebooks 00 + 03 have run (v2 prompts in production, golden dataset seeded).


## Module 7 · Live & automatic evaluation in production

🎯 *Outcome: synthetic "real users" hit the agent; 
- their traffic gets scored automatically
-  server-side by a managed judge
-  in-code by a reference-free scorer (trace-, observation- AND session-level)
and the worst failures flow back into the golden dataset.*

Production is different in one crucial way: **there is no expected_output**. 
Online evaluation therefore leans on reference-free signals:
- user feedback (M5)
- reference-free judges ("did this reply resolve the question?"),
- grounding checks against what the retriever fetched
- business rules.
  
Everything lands as scores on the **production environment**, separate from today's dev traffic.

In [ ]:
# ── 0.1 Get the code + the pinned stack (fresh Colab VM: clone first) ──
import os
if not os.path.isdir("../app"):                    # fresh Colab VM → clone the repo
    !git clone https://github.com/kartik-nighania/data-hack-summit-2026.git _workshop_repo
    %cd _workshop_repo/workshop
%pip install -q -r ../requirements.txt
print("✅ stack ready — if pip just upgraded packages, do Run ▸ Restart session once and rerun from the top.")


In [ ]:
import os, sys, json, time
from datetime import datetime, timedelta, timezone

sys.path.insert(0, os.path.abspath(".."))        # make the repo's app/ package importable

# Jupyter kernels already run an event loop; this lets libraries that call
# asyncio.run()/run_until_complete work inside notebook cells.
import nest_asyncio
nest_asyncio.apply()

print("✅ environment prepared |", sys.version.split()[0])


In [ ]:
# ── 0.2 Your API keys — paste them in, then run. Colab saves your notebook copy to Drive,
#        so this is a one-time edit per notebook (testing keys only — don't share the copy).
import os
os.environ["OPENAI_API_KEY"] = "sk-proj-..."          # platform.openai.com/api-keys
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."       # cloud.langfuse.com → Settings ▸ API Keys
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
os.environ["LANGFUSE_BASE_URL"] = "https://us.cloud.langfuse.com"   # EU account: https://cloud.langfuse.com

from app.config import load_keys
load_keys()   # validates; unedited placeholders fall back to .env or a prompt


In [ ]:
# ── 0.3 Constants (config.yaml) + the Langfuse client (PII masking hook registered) ─
from app.config import JUDGE_MODEL, INGESTION_WAIT_S, TRAFFIC_SESSIONS, get_lf
lf = get_lf()
if not lf.auth_check():
    raise SystemExit("❌ Langfuse authentication FAILED. Check keys + LANGFUSE_HOST region "
                     "(EU: https://cloud.langfuse.com / US: https://us.cloud.langfuse.com).")
LANGFUSE_HOST = os.environ["LANGFUSE_HOST"]
import importlib.metadata as _md
print("✅ Langfuse authenticated:", LANGFUSE_HOST)
for p in ["langfuse", "langchain", "langgraph", "deepeval", "openai", "fastmcp"]:
    print(f"   {p}=={_md.version(p)}")


In [ ]:
# Session imports: agent + feedback + the judge/DeepEval helpers taught in notebook 03
import asyncio
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from deepeval.test_case import LLMTestCase
from deepeval.metrics import ContextualRelevancyMetric
from app.agent import load_service_tools
from app.tools import ACCOUNT_TOOLS, POLICY_TOOLS
from app.evaluators import _norm
from app.judges import _track_judge, _mk, _measure
from app.golden import ensure_candidates_dataset
SERVICE_TOOLS = await load_service_tools()   # tool names feed the Module-8 usage chart


In [ ]:
# ── 7.1 Simulated production traffic (concurrent "users", sessions, feedback) ──
# lives in app/generate_fake_traffic.py (also headless: python -m app.generate_fake_traffic)
from app.generate_fake_traffic import generate_traffic, SESSION_LOG, TRAFFIC_STARTED_AT
traffic_v2 = await generate_traffic(TRAFFIC_SESSIONS, deploy_version="v2")
print(f"\n⏳ giving Langfuse {INGESTION_WAIT_S}s to ingest …"); time.sleep(INGESTION_WAIT_S)
prod = lf.api.trace.list(environment=["production"], limit=1, page=1)
print("production traces now queryable — filter the Tracing table: Environment = production")


### 7.2 Managed LLM-as-a-Judge on live traffic (server-side, no code, no ground truth)

UI (2 minutes): 
- **Evaluators ▸ Set up evaluator** → catalog template **Helpfulness** (or Relevance) → target
**Live data ▸ Observations** → filters: observation name = `meridian-support-agent` *(the root span — it
carries the whole request's input/output)*, environment = `production` → variable mapping from the live
preview (input → `question`, output → `answer`) → **sampling 100%** (real systems: 5–20%) → activate.

From now on every *new* production trace gets a `helpfulness` score minutes after it arrives.
To show: 
- filter Tracing to environment `langfuse-llm-as-a-judge` — the judge's own
executions, fully traced
- *backfill*: Tracing table → filter env=production → select rows → **Actions ▸
Evaluate** runs the evaluator on **already-collected** traffic (needs the evaluator's Fast Mode toggle).

## Running your own evaluator in a cronjob

In [ ]:
# ──  Your own online scorer: reference-free + component-level + backfill ──
class ResolutionVerdict(BaseModel):
    resolved: float = Field(ge=0, le=1, description="1.0 = fully resolved, professional, safe")
    reasoning: str

_RES_SYS = ("You review a housing-finance support reply WITHOUT knowing the correct answer. "
            "Score how well it RESOLVES the question. Anchors: 1.0 = contains the specific facts the "
            "question needs (amounts, dates, charges, document names, ticket ids) plus a clear next step; "
            "0.5 = addresses the topic but is missing some specifics; 0.3 or BELOW = vague reassurance "
            "with NO concrete figures, ids or steps ('it is all sorted, do not worry') - however friendly. "
            "Also penalize to 0.2 any rate promises or other customers' data. Judge substance, not length.")

async def score_recent_production(minutes_back=30, sample=12, since=None):
    """The 'online evaluation job' you would run on a schedule: pull recent prod traces,
    attach reference-free scores at TRACE level and component scores at OBSERVATION level."""
    since = since or (datetime.now(timezone.utc) - timedelta(minutes=minutes_back))  # SDK expects datetimes
    page = lf.api.trace.list(environment=["production"], from_timestamp=since, limit=50, page=1)
    traces = [t for t in page.data if t.name == "meridian-support-agent"][:sample]
    llm = ChatOpenAI(model=JUDGE_MODEL, temperature=0)
    scored = 0
    for t in traces:
        q = (t.input or {}).get("question", "") if isinstance(t.input, dict) else str(t.input)
        a = (t.output or {}).get("answer", "") if isinstance(t.output, dict) else str(t.output)
        out = await llm.with_structured_output(ResolutionVerdict, include_raw=True).ainvoke(
            [("system", _RES_SYS), ("user", f"QUESTION:\n{q}\n\nREPLY:\n{a}")])
        _track_judge(out["raw"]); v = out["parsed"]
        lf.create_score(trace_id=t.id, name="resolution_conf", value=round(v.resolved, 3),
                        comment=v.reasoning, environment="production")   # trace-level, reference-FREE
        obs = lf.api.observations.get_many(trace_id=t.id, limit=100)
         # Ticket found score
        ticket = any(o.name == "create_ticket" for o in obs.data)
        lf.create_score(trace_id=t.id, name="escalated", value=ticket, data_type="BOOLEAN",
                        comment="a service ticket was created in this conversation",
                        environment="production")
        retr = next((o for o in obs.data if o.name == "policy-kb-retriever"), None)
        if retr and a:
            rq = (retr.input or {}).get("query") if isinstance(retr.input, dict) else str(retr.input)
            chunks = (retr.output or {}).get("chunks") if isinstance(retr.output, dict) else None
            if rq and chunks:
                m = await asyncio.to_thread(_measure, _mk(ContextualRelevancyMetric),
                                            LLMTestCase(input=rq, actual_output=a, retrieval_context=chunks))
                lf.create_score(trace_id=t.id, observation_id=retr.id,          # ← OBSERVATION-level
                                name="retrieval_relevancy", value=round(m.score, 3),
                                comment=str(m.reason)[:300], environment="production")
        scored += 1
    lf.flush()
    return scored

n = await score_recent_production()
print(f"✅ scored {n} recent production traces: resolution_conf (trace) + escalated (trace, BOOLEAN)")
print("   + retrieval_relevancy attached to the policy-kb-retriever OBSERVATION where retrieval happened.")
print("   Open a scored policy trace → the retriever span carries its own score. That's component-level, live.")

## Running session level conversation evaluations 

In [ ]:
# ── 7.4 Session-level eval: multi-turn conversations, DeepEval-style ─────────
from deepeval.test_case import ConversationalTestCase, Turn, MultiTurnParams
from deepeval.metrics import ConversationalGEval

multi = [r for r in SESSION_LOG if len(r["turns"]) >= 2][:3]
conv_metric = ConversationalGEval(
    name="ConversationQuality",
    criteria=("Judge the WHOLE conversation: does the assistant stay consistent across turns, carry context "
              "from earlier turns into follow-ups, and remain professional and safe throughout?"),
    evaluation_params=[MultiTurnParams.CONTENT],
    model=JUDGE_MODEL, async_mode=False,
)
for r in multi:
    turns = []
    for t in r["turns"]:
        turns += [Turn(role="user", content=t["q"]), Turn(role="assistant", content=t["a"])]
    ctc = ConversationalTestCase(turns=turns)
    await asyncio.to_thread(_measure, conv_metric, ctc)
    lf.create_score(session_id=r["session_id"],                    # ← SESSION-level score
                    name="conversation_quality", value=round(conv_metric.score, 3),
                    comment=str(conv_metric.reason)[:350], environment="production")
    print(f"{r['session_id']}: conversation_quality={conv_metric.score:.2f}")
lf.flush()
print("\nOpen Tracing ▸ Sessions → these sessions now carry a conversation-level score —")
print("single-turn scores judge messages; THIS judges the conversation. (ConversationalTestCase = the multi-turn twin of LLMTestCase.)")

## Promote to dataset

In [ ]:
# ── 7.5 Close the loop: promote real production failures into the dataset ────
time.sleep(10)   # let the feedback + resolution scores land
CANDIDATES = ensure_candidates_dataset()   # staging set — the frozen v1 gate set stays untouched
cands = []
# OR pull from a timestamp and find if the trace is user scored
for r in SESSION_LOG:
    if r.get("deploy") != "v2": continue
    first = r["turns"][0]                      # the graded question/answer pair
    score = (0 if r.get("feedback") == 0 else 1) + (0 if not any(_norm(h) in _norm(first["a"]) for h in first["hints"]) else 1)
    cands.append((score, r, first))
cands.sort(key=lambda x: x[0])

promoted = []
for _score, r, first in cands[:2]:
    item_id = f"prod-fail-{first['trace_id'][:8]}"
    lf.create_dataset_item(
        dataset_name=CANDIDATES, id=item_id,
        input={"question": first["q"], "customer_id": r["customer_id"]},
        expected_output={"reference": "TODO: author the expected behaviour after review",
                         "must_mention": [], "expected_route": [], "expected_tools": []},
        metadata={"category": "prod_failure", "difficulty": "unknown", "policy_refs": []},
        source_trace_id=first["trace_id"],                     # ← the link back to the real incident
    )
    promoted.append(item_id)
print(f"✅ promoted {promoted} into '{CANDIDATES}' — it now has {len(lf.get_dataset(CANDIDATES).items)} items; the v1 gate set stays frozen.")
print("""
This is dataset-sourcing path #3 (from traces; the SDK way). The UI ways: any trace → 'Add to dataset' button
(single), or Tracing table → select rows → Actions → Add to dataset (batch, with field mapping). Path #4 is
review-driven: annotators processing the Module-6 queue can push reviewed traces to the dataset the same way.
Note the new items carry a source_trace_id → in the dataset UI each one links back to the original incident.
Today's incident is tomorrow's regression test — this loop is the whole point of online evaluation.
""")

## Module 8 · Custom dashboards & metrics

🎯 *Outcome: a production-monitoring dashboard in the UI, plus the same numbers pulled programmatically
through the Metrics API — including a custom business metric.*

**UI first (3 minutes), `Dashboards ▸ + New dashboard` → "Meridian Production"** — add 4 widgets:

| Widget | Data source | Metric | Filter/dimension |
|---|---|---|---|
| Cost per day | Observations | `totalCost · sum` by day | environment = production |
| User feedback trend | Scores (numeric) | `value · avg` of `user-feedback` by day | environment = production |


## Programatically fetch metrics using APIs

In [ ]:
# ── 8.1 The same numbers over the API (for YOUR tooling) ─────────────────────
# Budget note: the Metrics API allows 100 requests/DAY on the free tier - this module uses ~4.
import requests

AUTH = (os.environ["LANGFUSE_PUBLIC_KEY"], os.environ["LANGFUSE_SECRET_KEY"])
NOW = datetime.now(timezone.utc); FROM = NOW - timedelta(hours=6)

def metrics_query(q):
    """Resilient Metrics-API call: retries transient errors; explains 429 (100 req/day free tier)."""
    for attempt in (1, 2):
        r = requests.get(f"{LANGFUSE_HOST}/api/public/v2/metrics", auth=AUTH,
                         params={"query": json.dumps(q)}, timeout=30)
        if r.status_code == 429:
            print("⚠️ Metrics API daily budget exhausted (100 requests/day on the free tier) - "
                  "charts will be empty; they refill tomorrow. This is why monitor jobs batch queries!")
            return []
        if r.status_code >= 500 and attempt == 1:
            time.sleep(2); continue
        r.raise_for_status()
        return r.json().get("data", [])
    return []

# (a) production cost + observation count, hourly
cost_rows = metrics_query({
    "view": "observations",
    "metrics": [{"measure": "totalCost", "aggregation": "sum"}, {"measure": "count", "aggregation": "count"}],
    "dimensions": [], "filters": [{"column": "environment", "operator": "=", "value": "production", "type": "string"}],
    "timeDimension": {"granularity": "hour"},
    "fromTimestamp": FROM.strftime("%Y-%m-%dT%H:%M:%SZ"), "toTimestamp": NOW.strftime("%Y-%m-%dT%H:%M:%SZ")})

# (b) average user-feedback, hourly (scores-numeric view; BOOLEAN scores land here as 0/1)
fb_rows = metrics_query({
    "view": "scores-numeric",
    "metrics": [{"measure": "value", "aggregation": "avg"}, {"measure": "count", "aggregation": "count"}],
    "dimensions": [], "filters": [{"column": "name", "operator": "=", "value": "user-feedback", "type": "string"}],
    "timeDimension": {"granularity": "hour"},
    "fromTimestamp": FROM.strftime("%Y-%m-%dT%H:%M:%SZ"), "toTimestamp": NOW.strftime("%Y-%m-%dT%H:%M:%SZ")})

# (c) tool usage by observation name (client-side filter to our tools)
name_rows = metrics_query({
    "view": "observations", "metrics": [{"measure": "count", "aggregation": "count"}],
    "dimensions": [{"field": "name"}], "filters": [], "timeDimension": None,
    "fromTimestamp": FROM.strftime("%Y-%m-%dT%H:%M:%SZ"), "toTimestamp": NOW.strftime("%Y-%m-%dT%H:%M:%SZ")})
TOOL_NAMES = {t.name for t in ACCOUNT_TOOLS + POLICY_TOOLS + SERVICE_TOOLS}
tool_counts = {r["name"]: int(float(r["count_count"])) for r in name_rows if r.get("name") in TOOL_NAMES}
retriever_count = next((int(float(r["count_count"])) for r in name_rows if r.get("name") == "policy-kb-retriever"), 0)

# (d) trace counts: v2 metrics has no 'traces' view - the trace list's pagination metadata
# gives totals directly (the legacy GET /api/public/metrics/daily endpoint also works: it
# returns per-day countTraces/countObservations/totalCost, budgeted at 100 req/day).
prod_traces_total = lf.api.trace.list(environment=["production"], limit=1, page=1).meta.total_items

print("tool calls   :", dict(sorted(tool_counts.items(), key=lambda x: -x[1])))
print("retrievals   :", retriever_count, "(policy-kb-retriever spans)")
print("production traces so far:", prod_traces_total)

In [ ]:
# ── 8.2 A custom business metric + charts ────────────────────────────────────
# escalation_rate: share of production CONVERSATIONS that raised a ticket. A custom KPI is
# just YOUR definition computed over observable data - the traffic log knows each
# conversation's tool calls:
n_conversations = len(SESSION_LOG) or 1
n_escalated = sum(1 for r in SESSION_LOG
                  if any("create_ticket" in t.get("tools", []) for t in r["turns"]))
escalation_rate = n_escalated / n_conversations

# cross-check ticket VOLUME via the Metrics API, windowed to this session's traffic
esc_from = (TRAFFIC_STARTED_AT["t"] or FROM).strftime("%Y-%m-%dT%H:%M:%SZ")
esc_rows = metrics_query({
    "view": "observations", "metrics": [{"measure": "count", "aggregation": "count"}],
    "dimensions": [], "timeDimension": None,
    "filters": [{"column": "name", "operator": "=", "value": "create_ticket", "type": "string"},
                {"column": "environment", "operator": "=", "value": "production", "type": "string"}],
    "fromTimestamp": esc_from, "toTimestamp": NOW.strftime("%Y-%m-%dT%H:%M:%SZ")})
tickets_created = int(float(esc_rows[0]["count_count"])) if esc_rows else 0

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
hrs = [r["time_dimension"][-8:-3] if isinstance(r.get("time_dimension"), str) else str(r.get("time_dimension"))
       for r in cost_rows]
axes[0].bar(hrs, [float(r.get("sum_totalCost") or 0) for r in cost_rows]); axes[0].set_title("prod cost / hour ($)")
fh = [r["time_dimension"][-8:-3] if isinstance(r.get("time_dimension"), str) else "" for r in fb_rows]
axes[1].plot(fh, [float(r.get("avg_value") or 0) for r in fb_rows], marker="o"); axes[1].set_ylim(0, 1)
axes[1].set_title("avg user-feedback / hour")
axes[2].barh(list(tool_counts.keys()), list(tool_counts.values())); axes[2].set_title("tool calls (6h)")
plt.tight_layout(); plt.show()

print(f"escalation_rate (custom KPI): {n_escalated}/{n_conversations} conversations raised a ticket = {escalation_rate:.0%}")
print(f"ticket volume via Metrics API (since traffic start): {tickets_created} create_ticket calls")
print("Dashboard versions: widget on Observations (count, name=create_ticket) for volume - or chart the")
print("'escalated' BOOLEAN score from Module 7 (avg on Scores-numeric) for the rate. Your KPI, two roads.")